# Predict, then Select

Evaluate the released regression, threshold-classification, and joint models on complete single-CD16 candidate pools. The default uses all **1,370 within-slide test cells**, full-data models, and all three training seeds. The external option uses all **27,857 cells** from the batch-1 slide.

Select a GPU runtime for faster inference, then run the cells in order. Models and their validation-selected settings are fixed; this notebook performs no training. [Study methods](https://github.com/yyttim/predict-then-select/blob/main/docs/method.md).

In [ ]:
# Install the inference code used for this release.
import subprocess, sys
CODE_COMMIT = "78571c5c622f1c1ce664bbac6a59565a6c52689a"
assert len(CODE_COMMIT) == 40 and all(c in "0123456789abcdef" for c in CODE_COMMIT)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown==6.4.2",
    f"cellselect @ git+https://github.com/yyttim/predict-then-select.git@{CODE_COMMIT}"])

import hashlib, json, shutil, stat, tarfile, zipfile
from pathlib import Path, PurePosixPath
import gdown
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from cellselect.data import load_data
from cellselect.model import build_model
from cellselect.training import predict
from cellselect.metrics import prediction_metrics, selection_metrics

In [ ]:
# Choose the training budget, model-selection goal, and evaluation pool.
train_fraction = 1.0  # @param [0.25, 1.0] {type:"raw"}
goal = "purity"  # @param ["mean", "purity"]
pool = "test"  # @param ["test", "external"]
q = 0.10  # @param [0.05, 0.10, 0.20] {type:"raw"}
show_other_joint_head = True  # @param {type:"boolean"}
compare_reference = True  # @param {type:"boolean"}
show_plot = True  # @param {type:"boolean"}
ROOT = Path("marker_objectives_release").resolve()
MANIFEST_SHA256 = "978e968d5a030ad17c6c52c6f703798aec2666f9d96d4df51a9d3a2632fa624d"
ARTIFACTS = {
    "data": {"filename": "prepared-data.tar.xz", "prefix": "data",
             "id": "1NfFsoUeoDD6F-dpK-uySMBHXYRkaF3S-",
             "sha256": "713c8a075443d5e63a01f554d923e6f0bdfe9ea91f62c89c095f7d7afb9cb1c5"},
    "seed16": {"filename": "seed16.zip", "prefix": "models",
               "id": "1Z2pqshPMeXlH9G8hxCO0CEb1OPt9v3lG", "sha256": "65f64d2748aa4469d3427f6a46eeb9aa0b432b7fc4a811b25d524be53c065059"},
    "seed26": {"filename": "seed26.zip", "prefix": "models",
               "id": "1KSMtZB6r7EGsCOl5X6Hj3drlhibENgi0", "sha256": "f2ebd07690ce793a42a8e3c0d54066247676f0002aaa2cf1ace9d28aa4c1945a"},
    "seed36": {"filename": "seed36.zip", "prefix": "models",
               "id": "1-O5ZLtcVI9E2qKuMiMhUOjkKwy9VOkS-", "sha256": "1330d40695379c0f3ea468597883c3246c1cb66ab2f3ebae034568fd9a6fb157"},
    "reference": {"filename": "reference-results.zip", "prefix": "reference",
                  "id": "130_1DAII0U6Mi_R0u-Tl-UL-YjHO3QO9", "sha256": "659779f315b5e5b042b30034e6053b7b654bc84f2c105327e51650935b783df6"},
}
train_fraction, q = float(train_fraction), float(q)
assert train_fraction in (.25, 1.) and q in (.05, .1, .2)
assert goal in ("mean", "purity") and pool in ("test", "external")
packages = ["data", "seed16", "seed26", "seed36"] + (["reference"] if compare_reference else [])

In [ ]:
# Verify downloads and extract only regular files inside the release directory.
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def safe_path(name, prefix):
    relative = PurePosixPath(name)
    if (not relative.parts or relative.is_absolute() or ".." in relative.parts
            or "\\" in name or ":" in name or relative.parts[0] != prefix):
        raise ValueError(f"Unsafe archive path: {name}")
    path = ROOT.joinpath(*relative.parts).resolve()
    if not path.is_relative_to(ROOT):
        raise ValueError(f"Path leaves the release directory: {name}")
    return path

def copy_member(stream, path):
    # Shared metadata must agree across the three model packages.
    if path.parent == ROOT / "models" and path.name in ("selection.json", "study.json"):
        content = stream.read()
        if path.exists() and path.read_bytes() != content:
            raise ValueError(f"Conflicting release metadata: {path.name}")
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(content)
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        with path.open("wb") as output:
            shutil.copyfileobj(stream, output)

ROOT.mkdir(parents=True, exist_ok=True)
for key in packages:
    item = ARTIFACTS[key]
    assert len(item["sha256"]) == 64 and all(c in "0123456789abcdef" for c in item["sha256"])
    archive = ROOT / item["filename"]
    if not archive.exists():
        gdown.download(id=item["id"], output=str(archive), use_cookies=False, quiet=False)
    if sha256(archive) != item["sha256"]:
        raise ValueError(f"Download checksum mismatch: {archive.name}")
    if archive.name.endswith(".tar.xz"):
        with tarfile.open(archive, "r:xz") as bundle:
            for member in bundle:
                destination = safe_path(member.name, item["prefix"])
                if member.isdir():
                    continue
                if not member.isfile():
                    raise ValueError(f"Unsupported archive entry: {member.name}")
                with bundle.extractfile(member) as stream:
                    copy_member(stream, destination)
    else:
        with zipfile.ZipFile(archive) as bundle:
            for member in bundle.infolist():
                destination = safe_path(member.filename, item["prefix"])
                kind = stat.S_IFMT(member.external_attr >> 16)
                if kind not in (0, stat.S_IFREG, stat.S_IFDIR):
                    raise ValueError(f"Unsupported archive entry: {member.filename}")
                if not member.is_dir():
                    with bundle.open(member) as stream:
                        copy_member(stream, destination)
print("Downloads verified and extracted.")

In [ ]:
# Load the full pool and check the frozen model/data bindings.
assert sha256(ROOT / "data/manifest.json") == MANIFEST_SHA256
cohort = load_data(ROOT / "data", verify_hashes=True)
selection = json.loads((ROOT / "models/selection.json").read_text())
study = json.loads((ROOT / "models/study.json").read_text())
assert selection["data_manifest_sha256"] == study["data_manifest_sha256"] == MANIFEST_SHA256
assert selection["study_sha256"] == sha256(ROOT / "models/study.json")
tau = selection["threshold"]
assert tau == cohort.manifest["threshold"]["value"]
choices = [c for c in selection["selections"]
           if c["train_fraction"] == train_fraction and c["selected_for"] == goal]
assert len(choices) == 3 and {c["method"] for c in choices} == {"mse", "bce", "joint"}
choices.sort(key=lambda c: ("mse", "bce", "joint").index(c["method"]))
for choice in choices:
    assert [s["seed"] for s in choice["states"]] == [16, 26, 36]
    for state in choice["states"]:
        path = safe_path("models/" + state["checkpoint"], "models")
        assert sha256(path) == state["checkpoint_sha256"]
positions = cohort.rows(pool)
meta = cohort.metadata.iloc[positions].reset_index(drop=True)
images = np.asarray(cohort.images[positions], dtype=np.float32)
ids, target = meta.global_index.to_numpy(), meta.target.to_numpy(dtype=np.float64)
assert len(meta) == {"test": 1370, "external": 27857}[pool]
print(f"{pool}: {len(meta):,} cells; training fraction {train_fraction:g}; selection goal {goal}")
print(f"Fixed CD16 threshold: {tau:.6f}; retaining {int(np.ceil(q * len(meta))):,} cells")
display(pd.DataFrame([{k: c[k] for k in ("method", "weight", "step", "score")} for c in choices]))

In [ ]:
# Inference uses each checkpoint's original training normalization.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(4)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
cache, rows = {}, []
for choice in choices:
    for state in choice["states"]:
        relative = state["checkpoint"]
        if relative not in cache:
            checkpoint = torch.load(safe_path("models/" + relative, "models"),
                                    map_location="cpu", weights_only=True)
            config = checkpoint["config"]
            assert checkpoint["manifest_sha256"] == MANIFEST_SHA256 and checkpoint["threshold"] == tau
            assert checkpoint["step"] == choice["step"]
            assert (config["fraction"], config["method"], config["seed"], config["lambda_binary"]) == (
                train_fraction, choice["method"], state["seed"], choice["weight"])
            model = build_model().float().to(device)
            model.load_state_dict(checkpoint["model"], strict=True)
            outputs = predict(model, images, checkpoint["normalization"], device, batch_size=512)
            if choice["method"] == "mse":
                outputs.pop("hit_logit")
            elif choice["method"] == "bce":
                outputs.pop("mean")
            cache[relative] = outputs
            del model
        outputs = cache[relative]
        diagnostics = prediction_metrics(target, tau, **outputs)
        for name, score in outputs.items():
            primary = name == choice["score"]
            if not primary and not show_other_joint_head:
                continue
            values = selection_metrics(target, score, ids, tau, q)
            label = {"mse": "Regression", "bce": "Threshold classification", "joint": "Joint"}[choice["method"]]
            if choice["method"] == "joint":
                label += " (marker score)" if name == "mean" else " (probability score)"
            rows.append({"label": label, "method": choice["method"], "weight": choice["weight"],
                         "step": choice["step"], "seed": state["seed"], "score": name,
                         "primary_score": primary, **values, **diagnostics})
metrics = pd.DataFrame(rows)
print(f"Evaluated {len(cache)} distinct checkpoints on {device}; all three seeds retained.")

In [ ]:
# Mean ± sample SD across seeds. The alternate joint score uses the same checkpoint.
means = metrics.groupby("label", sort=False)[["mean_marker", "purity"]].mean()
sds = metrics.groupby("label", sort=False)[["mean_marker", "purity"]].std(ddof=1)
table = pd.DataFrame(index=means.index)
table["Mean selected CD16 (a.u.)"] = [f"{m:.2f} ± {s:.2f}" for m, s in zip(means.mean_marker, sds.mean_marker)]
table["Marker-high purity (%)"] = [f"{100*m:.2f} ± {100*s:.2f}" for m, s in zip(means.purity, sds.purity)]
display(table)
if compare_reference:
    reference = pd.read_csv(ROOT / f"reference/{pool}/metrics.csv", float_precision="round_trip")
    reference = reference.loc[(reference.train_fraction == train_fraction)
                              & (reference.selected_for == goal)
                              & np.isclose(reference.selection_fraction, q)]
    keys = ["method", "weight", "step", "seed", "score"]
    paired = metrics.merge(reference[[*keys, "mean_marker", "purity"]], on=keys,
                           suffixes=("", "_reference"), how="left", validate="one_to_one", indicator=True)
    assert paired._merge.eq("both").all(), "Missing matching reference state."
    display(pd.Series({name: (paired[name] - paired[name + "_reference"]).abs().max()
                       for name in ("mean_marker", "purity")}, name="Maximum absolute difference from reference"))
if show_plot:
    metric, scale = ("purity", 100) if goal == "purity" else ("mean_marker", 1)
    baseline = metrics.iloc[0]
    random = baseline.random_purity if goal == "purity" else baseline.random_mean
    ceiling = baseline.oracle_purity if goal == "purity" else baseline.oracle_mean
    fig, ax = plt.subplots(figsize=(7, 3.3))
    ax.errorbar(means[metric] * scale, np.arange(len(means)), xerr=sds[metric] * scale,
                fmt="o", color="#386b8b", capsize=3)
    ax.axvline(random * scale, color="0.6", linestyle="--", label="Random selection in expectation")
    ax.axvline(ceiling * scale, color="0.35", linestyle=":", label="Measured-outcome ceiling")
    ax.set(yticks=np.arange(len(means)), yticklabels=means.index,
           xlabel="Marker-high purity (%)" if goal == "purity" else "Mean selected CD16 (a.u.)")
    ax.set_title(f"{pool}: top {q:.0%} selection", loc="left")
    ax.invert_yaxis()
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=8, loc="lower right", bbox_to_anchor=(1, 1.02))
    plt.tight_layout()
    plt.show()

The goal determines which validation-selected checkpoint is loaded. The 5% and 20% selection budgets reuse the models chosen at 10%. Both joint scores use the same loaded weights, and measured markers enter evaluation only.

The table and error bars show variation across three trained seeds, not confidence intervals. The external slide uses the same numerical marker threshold and source normalization; its acquisition and measurement scale can differ. Small numerical differences across hardware or library versions may affect nearly tied selections.